# Übung 11: Bildbeschreibung und Sprachausgabe mit der Gemini API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/11_gemini_vision_tts.ipynb)

In dieser Übung lernst du, wie du die multimodalen Fähigkeiten der Gemini API nutzt, um:
1. Ein Bild zu analysieren und eine Beschreibung in deutscher Sprache zu generieren.
2. Diese Beschreibung mithilfe der Gemini Text-to-Speech (TTS) Funktionalität in eine Audiodatei umzuwandeln.

Diese Kombination ermöglicht es, Anwendungen zu bauen, die Bilder nicht nur verstehen, sondern sie auch für Benutzer "erzählen" können.

## 1. Installation und Setup

Zuerst installieren wir das notwendige SDK von Google.

In [ ]:
!pip install -U google-genai pillow requests

## 2. API-Konfiguration

Um die Gemini API nutzen zu können, benötigst du einen API-Key von [Google AI Studio](https://aistudio.google.com/).

In [ ]:
import os
from google import genai
from google.genai import types
from IPython.display import Image, Audio, display
import requests
from PIL import Image as PILImage
import io

# Gib hier deinen API-Key ein oder setze ihn als Umgebungsvariable
os.environ["GOOGLE_API_KEY"] = "DEIN_API_KEY"

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

## 3. Bild laden

Wir verwenden ein Beispielbild von Ultralytics (den bekannten Bus).

In [ ]:
image_url = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg"
image_response = requests.get(image_url)
image_bytes = image_response.content

# Bild zur Kontrolle anzeigen
display(Image(data=image_bytes))

## 4. Bildbeschreibung generieren

Wir nutzen das Modell `gemini-1.5-flash` (oder ein aktuelleres Modell wie `gemini-2.0-flash`), um das Bild auf Deutsch zu beschreiben.

In [ ]:
prompt = "Beschreibe dieses Bild detailliert auf Deutsch. Erwähne alle wichtigen Objekte und die Szene."

response = client.models.generate_content(
    model="gemini-1.5-flash",
    contents=[
        types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
        prompt
    ]
)

beschreibung = response.text
print("Generierte Beschreibung:")
print(beschreibung)

## 5. Sprachausgabe (Text-to-Speech)

Nun wandeln wir die generierte Beschreibung in Audio um. Wir nutzen dafür das Modell `gemini-1.5-flash-tts-preview-0830` oder ein entsprechendes TTS-fähiges Modell.
Hinweis: In der Vorschauversion der Gemini API können sich die Modellnamen für TTS häufig ändern.

In [ ]:
import wave

def save_wav(filename, pcm_data, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm_data)

tts_response = client.models.generate_content(
    model="gemini-2.0-flash-exp", # Oder ein spezifisches TTS-Modell falls verfügbar
    contents=f"Lies diesen Text natürlich vor: {beschreibung}",
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name='Kore')
            )
        ),
    )
)

audio_parts = [part for part in tts_response.candidates[0].content.parts if part.inline_data]
if audio_parts:
    audio_data = audio_parts[0].inline_data.data
    audio_file = "beschreibung_audio.wav"
    save_wav(audio_file, audio_data)
    print(f"Audio wurde unter {audio_file} gespeichert.")
    
    # Audio in der Übung abspielen
    display(Audio(audio_file))
else:
    print("Es wurde kein Audio generiert.")